In [1]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="ollama",
    model="qwen3:8b"
)

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n<think>\nOkay, let's tackle this. The user wants me to extract the most relevant context from the conversation history. The main goal here is to free up space by keeping only the essential information.\n\nFirst, the SESSION INTENT. The user started by asking about the capital of the moon, which the AI answered as Lunapolis. Then they asked about the weather, cheese miners, and a potential strike. The overall intent seems to be gathering information about Lunapolis, its weather, population, and labor situation. So the primary goal is to compile details about Lunapolis, including its capital, weather conditions, population of cheese miners, and the possibility of a strike.\n\nNext, the SUMMARY. The key points are the capital being Lunapolis, the weather data, 100,000 cheese miners, and the union strike possibility due to the new president. The reasoning for the strike is the miners' dissatisfaction. The

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

<think>
Okay, let's tackle this. The user wants me to extract the most relevant context from the conversation history. The main goal here is to free up space by keeping only the essential information.

First, the SESSION INTENT. The user started by asking about the capital of the moon, which the AI answered as Lunapolis. Then they asked about the weather, cheese miners, and a potential strike. The overall intent seems to be gathering information about Lunapolis, its weather, population, and labor situation. So the primary goal is to compile details about Lunapolis, including its capital, weather conditions, population of cheese miners, and the possibility of a strike.

Next, the SUMMARY. The key points are the capital being Lunapolis, the weather data, 100,000 cheese miners, and the union strike possibility due to the new president. The reasoning for the strike is the miners' dissatisfaction. The rejected options aren't mentioned here, so

## Trim/delete messages

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='6c7ee817-a0ba-4175-8c7c-bae1f7c2ca64'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='d4914b03-be51-4313-9a7d-dab08f91021f', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8120d3f7-0b64-4db7-ae27-ed59b785d7c7'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='db45f21c-e2fc-478f-b539-4d8653bfeb6a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='49f159b0-7a12-4ca1-bec3-41afa3519278'),
              AIMessage(content='<think>\nOkay, the user\'s device isn\'t turning on. They said

In [11]:
print(response["messages"][-1].content)

<think>
Okay, the user's device isn't turning on. They said it's plugged in and turned on, but maybe they mean it's not powering up. Wait, the user might have confusion in their wording. Let me check the previous messages.

First, the user asked, "My device won't turn on. What should I do?" Then the assistant asked, "Is the device plugged in and turned on?" The user replied, "Yes, it's plugged in and turned on." Hmm, that's a bit confusing. If the device isn't turning on, how is it plugged in and turned on? Maybe they meant it's plugged in but not powering up. Or maybe they're using "turned on" incorrectly. 

Next, the assistant asked, "Is the device showing any lights or indicators?" The user then asked, "What's the temperature of the device?" So the user is shifting to temperature. Maybe they're concerned the device is overheating or too cold. But if the device isn't turning on, temperature might not be the immediate issue. However, sometimes extreme temperatures can cause hardware f